# GLD/GDX Pairs Trading: Cointegration & Kalman Filter Analysis

**Dataset:** GLD (SPDR Gold Trust ETF) and GDX (VanEck Gold Miners ETF)  
**Period:** 2010-01-01 to 2024-12-31  
**Source:** Yahoo Finance via `yfinance`

This notebook analyzes the cointegrated relationship between gold spot exposure (GLD) and gold mining equities (GDX). Both instruments are driven by gold prices, but GDX carries additional equity and operational risk premium — making the spread between them economically meaningful and potentially mean-reverting. The analysis proceeds from data cleaning through EDA, formal cointegration testing, static and dynamic (Kalman filter) spread construction, signal generation, and regime analysis.

In [ ]:
# Uncomment to install dependencies
# !pip install yfinance pandas numpy matplotlib seaborn statsmodels pykalman scipy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 1.4

np.random.seed(42)
print('Libraries loaded successfully.')

---
## Step 1: Data Collection & Cleaning

We pull daily OHLCV data for GLD and GDX from Yahoo Finance using `yfinance`. We keep only the **adjusted close** price, which accounts for corporate actions (splits, dividends). Both tickers are downloaded in a single call so they share the same date index from the start.

Cleaning decisions:
- **Forward fill** isolated NaN gaps (carries the last valid price forward — standard for ETF price data where gaps are typically one-day data vendor issues, not actual non-trading days)
- **Drop** any rows with NaNs that remain after forward fill (e.g., leading NaNs before either ETF had data)
- All decisions are documented in the missing value report below

In [ ]:
# --- Constants ---
TICKERS    = ['GLD', 'GDX']
START_DATE = '2010-01-01'
END_DATE   = '2024-12-31'

# --- Download ---
print(f"Downloading {TICKERS} from {START_DATE} to {END_DATE}...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)

# auto_adjust=True means 'Close' already reflects adjusted prices
adj_close = raw['Close'][TICKERS].copy()
adj_close.columns = ['GLD_adj', 'GDX_adj']

print(f"\nRaw download shape : {adj_close.shape}")
print(f"Date range         : {adj_close.index[0].date()}  →  {adj_close.index[-1].date()}")
print(f"\nFirst 5 rows:")
display(adj_close.head())
print(f"\nLast 5 rows:")
display(adj_close.tail())

In [ ]:
def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame summarising missing values per column."""
    return pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %'    : (df.isnull().sum() / len(df) * 100).round(4),
        'Total Rows'   : len(df)
    })

print("=== Missing Value Report — Raw Download ===")
display(missing_value_report(adj_close))

In [ ]:
def clean_price_series(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Clean a price DataFrame by forward-filling isolated NaN gaps
    then dropping any remaining NaNs (e.g., leading rows).

    Returns:
        cleaned DataFrame and a report dict documenting the transformations.
    """
    rows_before = len(df)
    nans_before = int(df.isnull().sum().sum())

    # Step 1: forward fill isolated gaps
    df_filled = df.ffill()
    nans_after_ffill = int(df_filled.isnull().sum().sum())
    filled_count = nans_before - nans_after_ffill

    # Step 2: drop any remaining NaNs (should only be leading rows)
    df_clean = df_filled.dropna()
    rows_after  = len(df_clean)
    rows_dropped = rows_before - rows_after

    report = {
        'rows_raw'       : rows_before,
        'nans_raw'       : nans_before,
        'nans_ffilled'   : filled_count,
        'rows_dropped'   : rows_dropped,
        'rows_final'     : rows_after,
        'nans_final'     : int(df_clean.isnull().sum().sum()),
    }
    return df_clean, report


adj_close_clean, clean_report = clean_price_series(adj_close)

print("=== Cleaning Summary ===")
for k, v in clean_report.items():
    print(f"  {k:<20}: {v}")

assert clean_report['nans_final'] == 0, "NaNs remain after cleaning — inspect data!"
print("\nAssertion passed: zero NaN values in cleaned DataFrame.")

In [ ]:
# Save checkpoints for reproducibility
os.makedirs('data', exist_ok=True)

adj_close.to_csv('data/gld_gdx_raw.csv')
adj_close_clean.to_csv('data/gld_gdx_clean.csv')

print("Saved:")
print("  data/gld_gdx_raw.csv   — raw download")
print("  data/gld_gdx_clean.csv — after forward-fill and dropna")
print(f"\nFinal clean DataFrame shape: {adj_close_clean.shape}")
print(f"Date range: {adj_close_clean.index[0].date()}  →  {adj_close_clean.index[-1].date()}")

---
## Step 2: Log-Price Transformation

We apply a natural log transform to both price series before any analysis. Three reasons:

1. **Variance stabilisation** — raw prices exhibit heteroskedasticity (variance grows with price level). Log prices have more homogeneous variance across time.
2. **Additivity of returns** — log returns `Δlog(P)` are additive across periods, making them the standard unit of analysis in financial econometrics.
3. **Cointegration prerequisite** — the Engle-Granger and Johansen tests assume log-linear relationships between price levels. Working in log space ensures the hedge ratio `β` in `log(GLD) = α + β·log(GDX) + ε` has a clean economic interpretation (elasticity).

We also compute log returns here since they are needed for ADF testing in Step 4.

In [ ]:
df = adj_close_clean.copy()

# Log prices
df['log_GLD'] = np.log(df['GLD_adj'])
df['log_GDX'] = np.log(df['GDX_adj'])

# Log returns (first difference of log prices = continuously compounded return)
df['ret_GLD'] = df['log_GLD'].diff()
df['ret_GDX'] = df['log_GDX'].diff()

# The first row has NaN returns — drop it from the returns columns only;
# we keep the full df intact and use .dropna() locally where needed.
returns = df[['ret_GLD', 'ret_GDX']].dropna()

print("Main DataFrame columns:", df.columns.tolist())
print(f"df shape             : {df.shape}")
print(f"returns shape        : {returns.shape}")
print("\nSample (last 5 rows):")
display(df.tail())

In [ ]:
# Quick descriptive stats on log prices and returns
print("=== Log Price Descriptive Statistics ===")
display(df[['log_GLD', 'log_GDX']].describe().round(4))

print("\n=== Log Return Descriptive Statistics ===")
display(returns.describe().round(6))